# Predictive Benchmark Notebook

This notebook follows the required controlled benchmarking workflow and single-notebook protocol.

- File: `predictive_benchmark.ipynb`
- Reproducibility: fixed random seed and explicit execution order
- Constraint: later tasks must build on Task 1 assumptions unless explicitly challenged

In [1]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

print(f"Random seed fixed at {SEED}")

Random seed fixed at 42


## Task 1

### 1. Plan
1. Read `dataset description.pdf` first and extract core documentation evidence on variable meaning, collection context, and temporal availability.
2. Load `hotel_bookings.csv` and inspect schema, dtypes, cardinality, and alignment with documentation-defined variables.
3. Quantify missingness and check data quality risks: impossible values, category inconsistencies, duplicates, and suspicious distributions.
4. Flag leakage and methodological risks by variable, explicitly separating documentation-derived evidence from data-inferred concerns.
5. Produce a reproducible preprocessing design (no modeling) with clear justifications and verification checks.

### 2. Risks
- Data leakage from post-outcome variables or variables unavailable at the intended prediction timestamp.
- Invalid evaluation design if temporal structure is ignored.
- Bias/generalization risk because data comes from two Portuguese hotels and a fixed period.
- Data quality risks from duplicated rows, sentinel missingness, or implausible numeric values.
- Unsupported assumptions when documentation is ambiguous or when observed patterns can have multiple explanations.

### 3. Implementation

In [2]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import re
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

print(f"Seed fixed at {SEED}")

Seed fixed at 42


In [3]:
# Step 1: Read documentation FIRST (as required)
NOTEBOOK_DIR = Path.cwd()
DOC_CANDIDATES = [
    NOTEBOOK_DIR / 'dataset_description.pdf',
    NOTEBOOK_DIR / 'dataset description.pdf',
]

doc_path = next((p for p in DOC_CANDIDATES if p.exists()), None)
if doc_path is None:
    raise FileNotFoundError('Documentation file not found. Expected one of: dataset_description.pdf, dataset description.pdf (same folder as notebook).')

from pypdf import PdfReader
reader = PdfReader(str(doc_path))
doc_text = "\n".join(page.extract_text() or "" for page in reader.pages)
compact_doc = re.sub(r'\s+', '', doc_text.lower())

# Robust checks (insensitive to whitespace/newline extraction artifacts)
doc_evidence_phrases = {
    'context_two_hotels_portugal': ('bothhotelsarelocatedinportugal' in compact_doc),
    'target_is_canceled': ('iscanceled' in compact_doc),
    'anti_leakage_design': ('leakageoffutureinformation' in compact_doc),
    'as_of_day_prior_arrival': ('daypriortoeachbooking' in compact_doc),
    'null_not_applicable_note': ('null' in compact_doc and 'notapplicable' in compact_doc),
}

print(f"Documentation loaded: {doc_path.name}")
print(f"Pages: {len(reader.pages)} | Extracted characters: {len(doc_text):,}")
print('Evidence phrase checks:')
print(pd.Series(doc_evidence_phrases))

Documentation loaded: dataset description.pdf
Pages: 9 | Extracted characters: 20,807
Evidence phrase checks:
context_two_hotels_portugal    True
target_is_canceled             True
anti_leakage_design            True
as_of_day_prior_arrival        True
null_not_applicable_note       True
dtype: bool


In [4]:
# Step 2: Load dataset for analysis (same-folder, relative paths only)
DATA_CANDIDATES = [
    NOTEBOOK_DIR / 'hotel_bookings.csv',
    NOTEBOOK_DIR / 'hotel_bookings (1).csv',
    NOTEBOOK_DIR / 'hotel_bookings.xlsx',
    NOTEBOOK_DIR / 'hotel_bookings (1).xlsx',
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Dataset file not found in notebook folder. Expected one of: hotel_bookings.csv, hotel_bookings (1).csv, hotel_bookings.xlsx, hotel_bookings (1).xlsx.')

if data_path.suffix.lower() == '.csv':
    df = pd.read_csv(data_path)
elif data_path.suffix.lower() in {'.xlsx', '.xls'}:
    df = pd.read_excel(data_path)
else:
    raise ValueError(f'Unsupported dataset format: {data_path.suffix}')

print(f"Dataset loaded: {data_path.name}")
print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
df.head(3)

Dataset loaded: hotel_bookings (1).xlsx
Shape: (119390, 32)
Columns (32): ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [5]:
# Step 3: Schema + variable interpretation (documentation-derived, mapped to dataset column names)
doc_variable_notes = {
    'hotel': ('Hotel type (Resort/City)', 'Documentation-derived', 'Observed at booking context'),
    'is_canceled': ('Target: booking canceled (1) vs not canceled (0)', 'Documentation-derived', 'Outcome variable'),
    'lead_time': ('Days between booking creation and arrival', 'Documentation-derived', 'Pre-arrival, computed from booking and arrival dates'),
    'arrival_date_year': ('Arrival year', 'Documentation-derived', 'Pre-arrival date attribute'),
    'arrival_date_month': ('Arrival month', 'Documentation-derived', 'Pre-arrival date attribute'),
    'arrival_date_week_number': ('Arrival week number', 'Documentation-derived', 'Pre-arrival date attribute'),
    'arrival_date_day_of_month': ('Arrival day of month', 'Documentation-derived', 'Pre-arrival date attribute'),
    'stays_in_weekend_nights': ('Weekend nights in stay', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'stays_in_week_nights': ('Week nights in stay', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'adults': ('Number of adults', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'children': ('Number of children', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'babies': ('Number of babies', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'meal': ('Meal package (SC/BB/HB/FB/Undefined)', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'country': ('Country of origin (ISO 3-letter code)', 'Documentation-derived', 'Pre-arrival customer attribute'),
    'market_segment': ('Market segment designation', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'distribution_channel': ('Booking distribution channel', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'is_repeated_guest': ('Whether guest has prior profile history', 'Documentation-derived', 'Pre-arrival, derived from prior profile creation timing'),
    'previous_cancellations': ('Customer prior canceled bookings count', 'Documentation-derived', 'Pre-arrival historical feature'),
    'previous_bookings_not_canceled': ('Customer prior non-canceled bookings count', 'Documentation-derived', 'Pre-arrival historical feature'),
    'reserved_room_type': ('Reserved room type code', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'assigned_room_type': ('Assigned room type code', 'Documentation-derived', 'Potentially close-to-arrival operational attribute'),
    'booking_changes': ('Count of booking amendments before check-in/cancellation', 'Documentation-derived', 'May evolve over lifecycle; horizon-sensitive'),
    'deposit_type': ('No Deposit / Non Refund / Refundable derived from payments', 'Documentation-derived', 'Uses payments before arrival/cancellation; horizon-sensitive'),
    'agent': ('Travel agency identifier', 'Documentation-derived', 'Categorical ID; NULL may be not applicable'),
    'company': ('Company/entity identifier', 'Documentation-derived', 'Categorical ID; NULL may be not applicable'),
    'days_in_waiting_list': ('Days on waiting list before confirmation', 'Documentation-derived', 'Lifecycle-dependent, may be unavailable at booking creation'),
    'customer_type': ('Contract/Group/Transient/Transient-Party', 'Documentation-derived', 'Pre-arrival booking type'),
    'adr': ('Average Daily Rate', 'Documentation-derived', 'Price-related; may reflect transactional adjustments'),
    'required_car_parking_spaces': ('Required parking spaces', 'Documentation-derived', 'Pre-arrival booking attribute'),
    'total_of_special_requests': ('Number of special requests', 'Documentation-derived', 'May accumulate pre-arrival; horizon-sensitive'),
    'reservation_status': ('Final reservation status (Canceled/Check-Out/No-Show)', 'Documentation-derived', 'Post-outcome -> leakage risk'),
    'reservation_status_date': ('Date of final reservation status', 'Documentation-derived', 'Post-outcome -> leakage risk'),
}

schema_rows = []
for col in df.columns:
    desc, source, temporal = doc_variable_notes.get(col, ('Not found in manual map', 'Unknown', 'Unknown'))
    schema_rows.append({
        'column': col,
        'dtype': str(df[col].dtype),
        'n_unique': int(df[col].nunique(dropna=True)),
        'missing_pct': round(df[col].isna().mean() * 100, 2),
        'interpretation': desc,
        'interpretation_source': source,
        'temporal_availability_note': temporal,
    })

schema_df = pd.DataFrame(schema_rows).sort_values('column').reset_index(drop=True)
schema_df.head(12)

,column,dtype,n_unique,missing_pct,interpretation,interpretation_source,temporal_availability_note
0,adr,float64,8879,0.00,Average Daily Rate,Documentation-derived,Price-related; may reflect transactional adjus...
1,adults,int64,14,0.00,Number of adults,Documentation-derived,Pre-arrival booking attribute
2,agent,float64,333,13.69,Travel agency identifier,Documentation-derived,Categorical ID; NULL may be not applicable
3,arrival_date_day_of_month,int64,31,0.00,Arrival day of month,Documentation-derived,Pre-arrival date attribute
4,arrival_date_month,object,12,0.00,Arrival month,Documentation-derived,Pre-arrival date attribute
5,arrival_date_week_number,int64,53,0.00,Arrival week number,Documentation-derived,Pre-arrival date attribute
6,arrival_date_year,int64,3,0.00,Arrival year,Documentation-derived,Pre-arrival date attribute
7,assigned_room_type,object,12,0.00,Assigned room type code,Documentation-derived,Potentially close-to-arrival operational attri...
8,babies,int64,5,0.00,Number of babies,Documentation-derived,Pre-arrival booking attribute
9,booking_changes,int64,21,0.00,Count of booking amendments before check-in/ca...,Documentation-derived,May evolve over lifecycle; horizon-sensitive


In [6]:
# Step 4: Missing value quantification
missing_df = (
    df.isna()
      .sum()
      .rename('missing_count')
      .to_frame()
      .assign(missing_pct=lambda x: (x['missing_count'] / len(df) * 100).round(2))
      .sort_values(['missing_pct', 'missing_count'], ascending=False)
)

print('Top missingness columns:')
print(missing_df.head(10))

Top missingness columns:
                          missing_count  missing_pct
company                          112593        94.31
agent                             16340        13.69
country                             488         0.41
children                              4         0.00
hotel                                 0         0.00
is_canceled                           0         0.00
lead_time                             0         0.00
arrival_date_year                     0         0.00
arrival_date_month                    0         0.00
arrival_date_week_number              0         0.00


In [7]:
# Step 5: Data quality checks (impossible values, category consistency, duplicates, suspicious distributions)
quality_checks = {}

# Duplicates
quality_checks['duplicate_rows'] = int(df.duplicated().sum())
quality_checks['duplicate_row_pct'] = round(df.duplicated().mean() * 100, 2)

# Parse arrival date robustness
arrival_date = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'].astype(str) + '-' +
    df['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)
quality_checks['invalid_arrival_date_rows'] = int(arrival_date.isna().sum())

# Binary columns should be {0,1}
for col in ['is_canceled', 'is_repeated_guest']:
    invalid = (~df[col].isin([0, 1]) & df[col].notna()).sum()
    quality_checks[f'invalid_binary_{col}'] = int(invalid)

# Non-negative columns
non_negative_cols = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies',
    'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests'
]
for col in non_negative_cols:
    quality_checks[f'negative_{col}'] = int((df[col] < 0).fillna(False).sum())

# Calendar consistency
valid_months = {
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
}
quality_checks['invalid_arrival_month_values'] = int((~df['arrival_date_month'].isin(valid_months)).sum())
quality_checks['invalid_arrival_week_number'] = int(((df['arrival_date_week_number'] < 1) | (df['arrival_date_week_number'] > 53)).sum())
quality_checks['invalid_arrival_day_of_month'] = int(((df['arrival_date_day_of_month'] < 1) | (df['arrival_date_day_of_month'] > 31)).sum())

# Country code format (3 letters) among non-missing
country_non_null = df['country'].dropna().astype(str)
quality_checks['country_not_3_chars'] = int((country_non_null.str.len() != 3).sum())

# Category normalization check (whitespace/case variants)
def normalization_gap(series):
    raw = set(series.dropna().astype(str).unique())
    normalized = set(series.dropna().astype(str).str.strip().str.upper().unique())
    return len(raw), len(normalized), len(raw) - len(normalized)

cat_cols = ['meal', 'market_segment', 'distribution_channel', 'deposit_type', 'customer_type', 'reservation_status', 'hotel']
norm_rows = []
for c in cat_cols:
    raw_n, norm_n, reduced_by = normalization_gap(df[c])
    norm_rows.append({'column': c, 'raw_unique': raw_n, 'normalized_unique': norm_n, 'reduced_by_normalization': reduced_by})
category_consistency_df = pd.DataFrame(norm_rows)

# Expected documented category checks for key fields
expected_sets = {
    'deposit_type': {'No Deposit', 'Non Refund', 'Refundable'},
    'reservation_status': {'Canceled', 'Check-Out', 'No-Show'},
    'customer_type': {'Contract', 'Group', 'Transient', 'Transient-Party'},
    'meal': {'Undefined', 'SC', 'BB', 'HB', 'FB'},
}
unexpected_rows = []
for col, exp in expected_sets.items():
    observed = set(df[col].dropna().astype(str).str.strip().unique())
    unexpected = sorted(observed - exp)
    unexpected_rows.append({'column': col, 'unexpected_values': unexpected, 'n_unexpected': len(unexpected)})
unexpected_category_df = pd.DataFrame(unexpected_rows)

# Suspicious distribution indicators
df_tmp = df.copy()
df_tmp['total_stay_nights'] = df_tmp['stays_in_weekend_nights'] + df_tmp['stays_in_week_nights']
quality_checks['zero_total_stay_nights'] = int((df_tmp['total_stay_nights'] == 0).sum())
quality_checks['adr_negative_values'] = int((df_tmp['adr'] < 0).sum())
quality_checks['adr_above_99_9pct'] = int((df_tmp['adr'] > df_tmp['adr'].quantile(0.999)).sum())
quality_checks['lead_time_above_99_9pct'] = int((df_tmp['lead_time'] > df_tmp['lead_time'].quantile(0.999)).sum())
quality_checks['adults_eq_0_with_children_or_babies'] = int(((df_tmp['adults'] == 0) & ((df_tmp['children'].fillna(0) > 0) | (df_tmp['babies'] > 0))).sum())

quality_df = pd.Series(quality_checks).rename('count').to_frame()
print('Quality issue counts:')
print(quality_df)
print('\nCategory normalization consistency (raw vs normalized unique counts):')
print(category_consistency_df)
print('\nUnexpected categories against documented sets:')
print(unexpected_category_df)


Quality issue counts:
                                           count
duplicate_rows                           31994.0
duplicate_row_pct                           26.8
invalid_arrival_date_rows                    0.0
invalid_binary_is_canceled                   0.0
invalid_binary_is_repeated_guest             0.0
negative_lead_time                           0.0
negative_stays_in_weekend_nights             0.0
negative_stays_in_week_nights                0.0
negative_adults                              0.0
negative_children                            0.0
negative_babies                              0.0
negative_previous_cancellations              0.0
negative_previous_bookings_not_canceled      0.0
negative_booking_changes                     0.0
negative_days_in_waiting_list                0.0
negative_adr                                 1.0
negative_required_car_parking_spaces         0.0
negative_total_of_special_requests           0.0
invalid_arrival_month_values                 0.

In [8]:
# Step 6: Leakage and methodological risk flags
risk_flags = pd.DataFrame([
    {
        'variable': 'reservation_status',
        'risk_level': 'High',
        'risk_type': 'Data leakage',
        'evidence': 'Documentation-derived: defined as final reservation status (Canceled/Check-Out/No-Show), which occurs after outcome realization.',
        'action': 'Exclude from predictors.'
    },
    {
        'variable': 'reservation_status_date',
        'risk_level': 'High',
        'risk_type': 'Data leakage',
        'evidence': 'Documentation-derived: date of final status; temporally post-outcome.',
        'action': 'Exclude from predictors.'
    },
    {
        'variable': 'booking_changes',
        'risk_level': 'Medium',
        'risk_type': 'Horizon mismatch',
        'evidence': 'Documentation-derived: accumulates amendments up to check-in/cancellation; may be unavailable for early-horizon prediction.',
        'action': 'Include only if prediction timestamp is close to arrival and definition is explicit.'
    },
    {
        'variable': 'days_in_waiting_list',
        'risk_level': 'Medium',
        'risk_type': 'Horizon mismatch',
        'evidence': 'Documentation-derived: depends on waiting-list confirmation process over time.',
        'action': 'Treat as time-dependent; validate timestamp availability.'
    },
    {
        'variable': 'deposit_type',
        'risk_level': 'Medium',
        'risk_type': 'Potential target-proximal signal',
        'evidence': 'Documentation-derived: computed from payments before arrival/cancellation date.',
        'action': 'Use with caution; tie to explicit prediction timestamp.'
    },
    {
        'variable': 'assigned_room_type',
        'risk_level': 'Medium',
        'risk_type': 'Operational-postprocessing risk',
        'evidence': 'Documentation-derived + inferred: may reflect operational updates near arrival.',
        'action': 'Retain only for near-arrival use-cases; otherwise drop.'
    },
    {
        'variable': 'agent, company',
        'risk_level': 'Low',
        'risk_type': 'Missingness interpretation risk',
        'evidence': 'Documentation-derived: NULL can mean not applicable, not random missing.',
        'action': 'Encode as explicit category such as NOT_APPLICABLE.'
    },
    {
        'variable': 'all variables',
        'risk_level': 'Medium',
        'risk_type': 'Evaluation bias risk',
        'evidence': 'Documentation-derived context: two hotels in Portugal (2015-07-01 to 2017-08-31); inferred limited external validity.',
        'action': 'Use temporal validation and clearly scope claims.'
    },
])

risk_flags

,variable,risk_level,risk_type,evidence,action
0,reservation_status,High,Data leakage,Documentation-derived: defined as final reserv...,Exclude from predictors.
1,reservation_status_date,High,Data leakage,Documentation-derived: date of final status; t...,Exclude from predictors.
2,booking_changes,Medium,Horizon mismatch,Documentation-derived: accumulates amendments ...,Include only if prediction timestamp is close ...
3,days_in_waiting_list,Medium,Horizon mismatch,Documentation-derived: depends on waiting-list...,Treat as time-dependent; validate timestamp av...
4,deposit_type,Medium,Potential target-proximal signal,Documentation-derived: computed from payments ...,Use with caution; tie to explicit prediction t...
5,assigned_room_type,Medium,Operational-postprocessing risk,Documentation-derived + inferred: may reflect ...,Retain only for near-arrival use-cases; otherw...
6,"agent, company",Low,Missingness interpretation risk,Documentation-derived: NULL can mean not appli...,Encode as explicit category such as NOT_APPLIC...
7,all variables,Medium,Evaluation bias risk,Documentation-derived context: two hotels in P...,Use temporal validation and clearly scope claims.


### 4. Verification
Verification checks performed:
- Documentation-read check: key phrases on anti-leakage extraction design, two-hotel context, and NULL semantics were found in extracted PDF text.
- Schema check: dataset columns, dtypes, and missingness profile were computed without errors.
- Data quality check: impossible value and consistency checks executed and quantified.
- Methodology check: no modeling, no training, and no external data used.

If any check had failed (missing files, parse errors, invalid schema assumptions), Task 1 would be revised before finalizing.

### 5. Revised final answer
1. **Dataset schema understanding**
- **Documentation-derived:** The dataset represents hotel booking demand/cancellation records from two Portuguese hotels; each row is a booking observation.
- **Data-inferred:** Loaded `hotel_bookings.csv` has 32 columns and a mixed schema (numeric + categorical + date-like strings).

2. **Variable interpretation based on the documentation**
- **Documentation-derived:** `is_canceled` is the cancellation target; `reservation_status` and `reservation_status_date` are final-status variables and temporally post-outcome.
- **Documentation-derived:** Several features are engineered and timestamped relative to day prior to arrival, which matters for temporal availability assumptions.
- **Data-inferred:** Column names in CSV map cleanly to the documented variables (snake_case equivalents).

3. **Missing value assessment**
- **Data-inferred:** Missingness is highly concentrated in `company` (very high) and `agent` (moderate), with minor missingness in `country` and near-zero elsewhere.
- **Documentation-derived:** For `agent`/`company`, NULL may indicate *not applicable* rather than unknown.

4. **Data quality issues detected**
- **Data-inferred:** Substantial exact duplicate rows are present and require explicit handling policy.
- **Data-inferred:** Additional quality indicators were quantified: invalid calendar fields, binary violations, negative values for non-negative fields, category normalization gaps, and tail anomalies (e.g., extreme `adr`, extreme `lead_time`, zero-night stays).
- **Uncertainty note:** Without a booking ID, duplicates cannot be safely interpreted as guaranteed errors.

5. **Leakage and methodological risk flags**
- **High-risk leakage:** `reservation_status`, `reservation_status_date` (drop).
- **Horizon-sensitive risk:** `booking_changes`, `days_in_waiting_list`, `deposit_type`, `assigned_room_type`, `total_of_special_requests` depending on prediction timestamp.
- **Generalization/evaluation risk:** Two-hotel, one-country, 2015–2017 context limits external validity; temporal splitting is preferable to random splitting.

6. **Recommended preprocessing strategy**
- Define prediction timestamp explicitly before feature selection.
- Remove high-leakage variables (`reservation_status`, `reservation_status_date`).
- Treat `agent`/`company` missing as explicit category (e.g., `NOT_APPLICABLE`) per documentation.
- Treat `country` missing as `UNKNOWN`; keep a missingness indicator if needed.
- Standardize category strings (strip whitespace, consistent casing).
- Parse arrival date fields into a single date and prefer temporal train/validation/test split.
- Keep a duplicate-policy branch: report results with and without exact duplicates rather than silently dropping.
- Handle extreme numeric tails via robust capping/winsorization only after documenting thresholds.

7. **Verification and uncertainties**
- All Task 1 code ran without errors and stayed within constraints (no modeling).
- Documentation-derived conclusions are explicitly separated from data-inferred conclusions.
- Remaining uncertainty: exact business semantics of duplicates and some lifecycle-dependent variables relative to the final modeling horizon.

## Task 2

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 3

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 4

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 5

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 6

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 7

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.